# Day 030 — Exercise 2: chain_steps

**What you'll build:** `chain_steps(steps, stop_on_error=True) -> list` — runs a list of `(name, fn)` tuples in sequence using `run_step`. When a step fails and `stop_on_error=True`, remaining steps get status='skipped' without calling their `fn`. The returned list always has exactly `len(steps)` records.

**Why it matters:** Downstream steps often depend on upstream output. Skipping rather than running them prevents corrupt state when the data they would have received is missing or broken.

In [ ]:
import time

## Provided: run_step

In [ ]:
import time


def run_step(name: str, fn) -> dict:
    start = time.time()
    try:
        result = fn()
        return {
            "name":       name,
            "status":     "ok",
            "result":     result,
            "error":      None,
            "duration_s": round(time.time() - start, 3),
        }
    except Exception as e:
        return {
            "name":       name,
            "status":     "error",
            "result":     None,
            "error":      str(e),
            "duration_s": round(time.time() - start, 3),
        }

## Your Implementation

In [ ]:
def chain_steps(steps: list, stop_on_error: bool = True) -> list:
    """
    Run a list of (name, fn) tuples; skip tail on error if stop_on_error.

    Args:
        steps:         List of (name, fn) tuples.
        stop_on_error: If True, skip remaining steps after first failure.

    Returns:
        List of step result dicts — always len(steps) records.
        Skipped steps have status='skipped', result=None, error=None, duration_s=0.0.
    """
    # TODO: results = []; failed = False
    # TODO: for name, fn in steps:
    #     if failed and stop_on_error: append skipped record
    #     else: step_result = run_step(name, fn); append it
    #           if step_result['status'] == 'error': failed = True
    # TODO: return results
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    def _ok(val):
        def fn(): return val
        return fn
    def _fail():
        raise RuntimeError('step failed')

    # Check 1: defined
    try:
        assert 'chain_steps' in globals()
        passed += 1; print('\u2705 Check 1: chain_steps defined')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}')
        return

    # Check 2: all steps pass → list of 3 ok records
    try:
        steps = [('a', _ok(1)), ('b', _ok(2)), ('c', _ok(3))]
        results = chain_steps(steps)
        assert isinstance(results, list), \
            f'expected list, got {type(results)}'
        assert len(results) == 3, \
            f'expected 3 records, got {len(results)}'
        assert all(r['status'] == 'ok' for r in results), \
            f'all should be ok: {[r["status"] for r in results]}'
        passed += 1; print('\u2705 Check 2: all-ok → 3 ok records')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: step_b fails → step_c is skipped (stop_on_error=True)
    try:
        steps = [('step_a', _ok('good')), ('step_b', _fail), ('step_c', _ok('done'))]
        results = chain_steps(steps, stop_on_error=True)
        assert len(results) == 3, \
            f'expected 3 records, got {len(results)}'
        assert results[0]['status'] == 'ok', \
            f"step_a should be 'ok': {results[0]['status']!r}"
        assert results[1]['status'] == 'error', \
            f"step_b should be 'error': {results[1]['status']!r}"
        assert results[2]['status'] == 'skipped', \
            f"step_c should be 'skipped': {results[2]['status']!r}"
        passed += 1; print('\u2705 Check 3: fail → remaining steps skipped')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: stop_on_error=False → all steps run even after failure
    try:
        steps = [('step_a', _ok('good')), ('step_b', _fail), ('step_c', _ok('done'))]
        results = chain_steps(steps, stop_on_error=False)
        assert results[1]['status'] == 'error', \
            f"step_b should still be 'error': {results[1]['status']!r}"
        assert results[2]['status'] == 'ok', \
            f"step_c should run and be 'ok': {results[2]['status']!r}"
        passed += 1; print('\u2705 Check 4: stop_on_error=False → all steps run')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: skipped record has correct sentinel values
    try:
        steps = [('a', _fail), ('b', _ok(1))]
        results = chain_steps(steps)
        skipped = results[1]
        assert skipped['status']     == 'skipped',  f"status: {skipped['status']!r}"
        assert skipped['result']     is None,       f"result: {skipped['result']!r}"
        assert skipped['error']      is None,       f"error: {skipped['error']!r}"
        assert skipped['duration_s'] == 0.0,        f"duration_s: {skipped['duration_s']}"
        assert skipped['name']       == 'b',        f"name: {skipped['name']!r}"
        passed += 1; print('\u2705 Check 5: skipped sentinel values correct')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def chain_steps(steps: list, stop_on_error: bool = True) -> list:
    results = []
    failed  = False
    for name, fn in steps:
        if failed and stop_on_error:
            results.append({
                "name":       name,
                "status":     "skipped",
                "result":     None,
                "error":      None,
                "duration_s": 0.0,
            })
        else:
            step_result = run_step(name, fn)
            results.append(step_result)
            if step_result["status"] == "error":
                failed = True
    return results
```

</details>